# KPI Analysis – Delivery Performance

This notebook analyzes delivery performance, including actual delivery time, on-time versus late deliveries, delay duration, and delivery reliability.

In [1]:
import pandas as pd
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)

Python: 3.13.15 (main, Sep  1 2026, 14:16:48) [MSC v.1944 64 bit (AMD64)]
Pandas: 3.0.5


In [2]:
from pathlib import Path

data_path = Path("../data")

csv_files = list(data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

In [3]:
from pathlib import Path
print(Path.cwd())

c:\Users\user777\Desktop\Conclusion Intelegence project\SpikupCapstone2026_CYRV\notebooks


In [4]:
data_path = Path("../data")

print("Data folder exists:", data_path.exists())

for item in data_path.iterdir():
    print(item.name)

Data folder exists: True
cyrv
README.md


In [5]:
data_path = Path("../data/cyrv")

csv_files = list(data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

CYRV_customers_dataset.csv
CYRV_geolocation_dataset.csv
CYRV_orders_dataset.csv
CYRV_order_items_dataset.csv
CYRV_order_payments_dataset.csv
CYRV_order_reviews_dataset.csv
CYRV_products_dataset.csv
CYRV_sellers_dataset.csv
product_category_name_translation.csv


In [6]:
file_info = []

for file in csv_files:
    file_info.append({
        "file": file.name,
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    })

pd.DataFrame(file_info).sort_values("size_mb", ascending=False)

,file,size_mb
1,CYRV_geolocation_dataset.csv,59.39
2,CYRV_orders_dataset.csv,16.93
3,CYRV_order_items_dataset.csv,14.83
5,CYRV_order_reviews_dataset.csv,13.78
0,CYRV_customers_dataset.csv,8.71
4,CYRV_order_payments_dataset.csv,5.61
6,CYRV_products_dataset.csv,2.30
7,CYRV_sellers_dataset.csv,0.17
8,product_category_name_translation.csv,0.00


In [7]:
customers = pd.read_csv(data_path / "CYRV_customers_dataset.csv")
orders = pd.read_csv(data_path / "CYRV_orders_dataset.csv")
order_items = pd.read_csv(data_path / "CYRV_order_items_dataset.csv")
payments = pd.read_csv(data_path / "CYRV_order_payments_dataset.csv")
reviews = pd.read_csv(data_path / "CYRV_order_reviews_dataset.csv")
products = pd.read_csv(data_path / "CYRV_products_dataset.csv")
sellers = pd.read_csv(data_path / "CYRV_sellers_dataset.csv")
category_translation = pd.read_csv(
    data_path / "product_category_name_translation.csv"
)

print("Datasets loaded successfully ✅")

Datasets loaded successfully ✅


In [8]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [9]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [10]:
print("First order:", orders["order_purchase_timestamp"].min())
print("Last order:", orders["order_purchase_timestamp"].max())

First order: 2016-09-04 21:15:19
Last order: 2018-10-17 17:30:18


In [11]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

delivered_orders["actual_delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivered_orders["on_time"] = (
    delivered_orders["order_delivered_customer_date"]
    <= delivered_orders["order_estimated_delivery_date"]
)

print("Delivered orders:", len(delivered_orders))
print(
    "Average actual delivery days:",
    round(delivered_orders["actual_delivery_days"].mean(), 2)
)
print(
    "On-time delivery rate:",
    round(delivered_orders["on_time"].mean() * 100, 2),
    "%"
)

Delivered orders: 96478
Average actual delivery days: 12.56
On-time delivery rate: 91.88 %


In [12]:
late_orders = delivered_orders[
    delivered_orders["on_time"] == False
].copy()

print("Late orders:", len(late_orders))

print(
    "Late delivery rate:",
    round(len(late_orders) / len(delivered_orders) * 100, 2),
    "%"
)

print(
    "Average delay days for late orders:",
    round(late_orders["delay_days"].mean(), 2)
)

print(
    "Maximum delay days:",
    round(late_orders["delay_days"].max(), 2)
)

Late orders: 7834
Late delivery rate: 8.12 %
Average delay days for late orders: 9.55
Maximum delay days: 188.98


In [13]:
late_orders["delay_days"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    7826.000000
mean        9.551776
std        13.952540
min         0.002500
50%         5.806481
75%        11.820978
90%        21.536088
95%        29.639578
99%        62.786780
max       188.975081
Name: delay_days, dtype: float64

In [14]:
print("Late orders with missing delay_days:")
display(
    late_orders[
        late_orders["delay_days"].isna()
    ][[
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]]
)

print("\nTop 10 longest delays:")
display(
    late_orders[
        ["order_id", "delay_days"]
    ]
    .sort_values("delay_days", ascending=False)
    .head(10)
)

Late orders with missing delay_days:


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,NaT,2018-07-19



Top 10 longest delays:


,order_id,delay_days
55619,1b3190b2dfa9d789e1f14c05b647a14a,188.975081
19590,ca07593549f1816d26a572e06dc1eab6,181.608785
11399,47b40429ed8cce3aee9199792275433f,175.869109
81401,2fe324febf907e3ea3f2aa9650869fa5,167.708414
89130,285ab9426d6982034523a855f55a885e,166.583380
61610,440d0d17af552815d15a9e41abe49359,165.633912
68769,c27815f7e3dd0b926b58552628481575,162.718345
40847,d24e8541128cea179a11a65176e0a96f,161.775336
38509,0f4519c5f1c541ddec9f21b3bddd533a,161.609965
54480,2d7561026d542c8dbd8f0daeadf67a43,159.609931


In [15]:
severe_delays = late_orders[late_orders["delay_days"] >= 30]

print("Orders delayed by 30 days or more:", len(severe_delays))

Orders delayed by 30 days or more: 360


## Initial Delivery Performance Findings

- Most delivered orders arrived on time, while a smaller share were delivered after the estimated delivery date.
- Among late orders, the median delay was approximately 5.8 days.
- A small number of orders experienced very long delays, with the maximum delay reaching approximately 189 days.
- 360 orders were delayed by 30 days or more, indicating a small group of severe delivery delays.